In [1]:
import pandas as pd

# Load the CSV file
df = pd.read_csv('/home/chen/repos/sol_lab/cosmo_10k.csv')



In [2]:
import networkx as nx

# Create a graph
G = nx.Graph()

# Add edges with attributes
for _, row in df.iterrows():
    G.add_edge(row['source'], row['target'], weight=row['weight'], distance=row['distance'])


## Clustering Methods ##

In [3]:
df_nodes = pd.DataFrame({'id': list(G.nodes())})

### Louvain Clustering ###

In [4]:
import community as community_louvain

# Apply Louvain clustering
louvain_partition = community_louvain.best_partition(G, weight='weight')

# Add Louvain clustering results to the DataFrame
df_nodes['louvain_cluster'] = df_nodes['id'].map(louvain_partition)

### Spectral Clustering ###

In [5]:
from sklearn.cluster import SpectralClustering
import numpy as np

# Create adjacency matrix
adj_matrix = nx.to_numpy_array(G, weight='weight')

# Apply Spectral Clustering
n_clusters = 5  # Specify the number of clusters
sc = SpectralClustering(n_clusters=n_clusters, affinity='precomputed')
spectral_labels = sc.fit_predict(adj_matrix)

# Assign clusters back to nodes
df_nodes['spectral_cluster'] = df_nodes['id'].apply(lambda node: spectral_labels[list(G.nodes()).index(node)])

### KMeans ###

In [6]:
from node2vec import Node2Vec
from sklearn.cluster import KMeans

# Generate embeddings using Node2Vec
node2vec = Node2Vec(G, dimensions=64, walk_length=30, num_walks=200, workers=4)
model = node2vec.fit(window=10, min_count=1, batch_words=4)

# Get embeddings for each node
embeddings = {node: model.wv[node] for node in G.nodes()}
X = np.array([embeddings[node] for node in G.nodes()])

# Apply KMeans clustering
kmeans = KMeans(n_clusters=n_clusters, random_state=0).fit(X)

# Add KMeans clustering results to the DataFrame
df_nodes['kmeans_cluster'] = df_nodes['id'].apply(lambda node: kmeans.labels_[list(G.nodes()).index(node)])


Computing transition probabilities:   0%|          | 0/9166 [00:00<?, ?it/s]

Generating walks (CPU: 1):  70%|███████   | 35/50 [00:40<00:17,  1.19s/it]

Generating walks (CPU: 2): 100%|██████████| 50/50 [01:03<00:00,  1.26s/it]


## Visualize clustering ##

In [7]:
df_nodes.to_csv('clustering_results.csv', index=False)